# Testing for allelic imbalance in control conditions in Desiree with short-read RNA-seq

In [ ]:
import polyase as poly
import pandas as pd
import RNApysoforms as RNApy
import polars as pl

In [ ]:
poly.__version__

## 1) Reading in the data 

In [ ]:
# Metadata for genes
var_obs_file = "potato-allelic-orthogroups/output_desiree_liftoff/genespace/De_v1.unitato_liftoff_haplotap_genespace_categories.tsv"
# directory of gene counts from oarfish
gene_counts_dir = "Desiree_ASE/output/nf_core_rnaseq_short_read_heat_phased_desiree_liftoff/salmon/"

tx2gene = "Desiree_ASE/output/nf_core_rnaseq_short_read_heat_phased_desiree_liftoff/salmon/tx2gene.tsv"
#nf-core-plantlongrnaseq/work/3f/6abd9f23b65ae2de98213bcc0ce3a5
# Define sample IDs and their conditions
sample_info = {
    "DE_C_H_13": "control",
    "DE_C_H_14": "control",
    "DE_C_H_15": "control",
    "DE_C_H_16": "control",
    "DE_H_H_17": "heat",
    "DE_H_H_18": "heat",
    "DE_H_H_19": "heat",
    "DE_H_H_20": "heat"
} 
# This can take some time (1 min per sample)
# With isoforms
# With isoforms
adata_isoform = poly.load_ase_data(
    var_obs_file=var_obs_file,
    sample_info=sample_info,
    tx_to_gene_file=tx2gene,
    isoform_counts_dir=gene_counts_dir,
    quant_dir=gene_counts_dir
)

### Add annotation

For interpretation of results we add functional gene annotations from an additional file to the adata.var

In [ ]:
# Load functional annotation data
annotation_file = f"reference/ADAPT_liftoff/Phureja_v4-v6.1_translations.xlsx"
df_annotation = pd.read_excel(annotation_file)
df_annotation = df_annotation.iloc[:, 1:3]
df_annotation.columns = ['gene_id_unitato', 'functional_annotation']

# Remove version suffix from gene IDs (e.g., "GENE.1" -> "GENE")
adata_isoform.var['gene_id_unitato'] = adata_isoform.var['gene_id'].str.rsplit('.', n=1).str[0]
adata_isoform.var['original_index'] = adata_isoform.var.index

# Merge annotations
merged_data = pd.merge(
    adata_isoform.var,
    df_annotation,
    on='gene_id_unitato',
    how='left'
)

# Remove duplicates and restore index
merged_data = merged_data.drop_duplicates(subset=['original_index', 'gene_id'], keep='first')
merged_data.index = merged_data['original_index']
merged_data = merged_data.drop(columns=['original_index'])

# Update AnnData object
adata_isoform.var = merged_data

Let's verify whether Bambu detected any previously unannotated transcripts in our dataset.

Now we will aggregate transcript-level counts to gene-level counts for allele-specific expression analysis. Unique counts are just summed. For ambgious counts we take the mean, under the asseumption that multimapping between transcripts of the same gene to other genes is similar.

In [ ]:
adata_gene = poly.aggregate_transcripts_to_genes(adata_isoform)

In [ ]:
#CPM threshold (automatically normalized)
adata_gene_filtered = adata_gene[(adata_gene.obs['condition'] == 'control'), :]
adata_gene_filtered = poly.filter_low_expressed_genes(
    adata_gene_filtered,
    min_expression=100.0,  # 20 counts
    mode= 'all',
    layer= "em_counts",
    #library_size_dependent=True, # this filters min_expression based on CPM of unique counts
    verbose=True)

#### Calculate allelic ratios and multimapping ratios

In [ ]:
adata_isoform_filtered = poly.calculate_allelic_ratios(adata_gene_filtered , 'unique_counts')
adata_isoform_filtered = poly.calculate_allelic_ratios(adata_gene_filtered , 'em_counts')
adata_isoform_filtered = poly.calculate_multi_ratios(adata_gene_filtered, 'unique_counts', 'ambiguous_counts')

#### Plotting of allelic ratios for quality control

In [ ]:
# filter transcripts that are not equal lengths
mask = adata_isoform_filtered.var["CDS_haplotype_with_longest_annotation"] == "equal_lengths"
adata_isoform_length_filtered = adata_isoform_filtered[:,mask].copy()
fig = poly.plot_allelic_ratios(
    adata_isoform_length_filtered,
    synteny_category="1hap1_1hap2_1hap3_1hap4_s", 
    sample='all', 
    ratio_type="both",
    figsize = (6,3),
    kde = True,
    multimapping_threshold=0.25,
    #save_path="allelic_ratios.svg"
)


After filtering for alleles of equal length, several of the highly biased genes are removed.

Another potential source of bias arises from unequal transcript numbers per gene. For example, a novel transcript may have been identified on only one haplotype. Therefore, we will filter to include only genes where all alleles have the same number of transcripts.

In [ ]:
# Group by syntelog ID and check for equal transcript numbers across alleles
synt_ids_same_tx_number = adata_isoform_filtered.var.groupby('Synt_id').aggregate({
    'n_transcripts': set
})

# Count genes with different transcript numbers before filtering
genes_diff_tx_number = synt_ids_same_tx_number[synt_ids_same_tx_number['n_transcripts'].apply(lambda x: len(x) > 1)]
print(f"Genes with different transcript numbers across alleles: {len(genes_diff_tx_number)}")

# Keep only syntelogs where all alleles have the same number of transcripts
synt_ids_same_tx_number = synt_ids_same_tx_number[synt_ids_same_tx_number['n_transcripts'].apply(lambda x: len(x) == 1)]
print(f"Genes with equal transcript numbers across alleles: {len(synt_ids_same_tx_number)}")

# Filter for equal-length transcripts with equal transcript numbers
mask = (adata_isoform_filtered.var["CDS_haplotype_with_longest_annotation"] == "equal_lengths") & \
       (adata_isoform_filtered.var['Synt_id'].isin(synt_ids_same_tx_number.index))



adata_isoform_length_filtered = adata_isoform_filtered[:, mask].copy()

# Plot allelic ratios after filtering
fig = poly.plot_allelic_ratios(
    adata_isoform_length_filtered,
    synteny_category="1hap1_1hap2_1hap3_1hap4_s",
    sample='all',
    ratio_type="both",
    figsize=(6, 3),
    kde=True,
    multimapping_threshold=0.25,
    #save_path="allelic_ratios.svg"
)

Now there are less alleles with ratios around 0 and >0.8.

In [ ]:
mask = (
    (adata_gene_filtered.var["multimapping_ratio"] < 0.90) &
    adata_gene_filtered.var["synteny_category"].str.contains("1hap1_1hap2_1hap3_1hap4_s") &
    (adata_gene_filtered.var['Synt_id'].isin(synt_ids_same_tx_number.index))
)
adata_gene_length_filter = adata_gene_filtered[:, mask].copy()


cis_results_control_em = poly.test_allelic_ratios_within_conditions(adata_gene_length_filter, 
                                                                 layer="em_counts", 
                                                                 test_condition= "control", 
                                                                 inplace=True)
# Get top differential syntelogs
cis_top_results_em = poly.get_top_differential_syntelogs(cis_results_control_em, 
                                                      n=45, sort_by='ratio_difference',
                                                      fdr_threshold=0.05, 
                                                      ratio_threshold=0.1)




In [ ]:
# export synt_id and multimapping ratio for genes with equal transcript numbers and equal lengths
adata_gene_length_filter.var[['Synt_id', 'multimapping_ratio', "n_transcripts"]].to_csv("LR_DESIREE_PAPER/ANALYSIS/final_tables_paper/short_read_multimapping_counts.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Filter for syntelogs with 4 alleles
cis_results_control_4alleles = cis_results_control_em[cis_results_control_em['n_alleles'] == 4]

# Add chromosome information from the gene_id
cis_results_control_4alleles['chromosome'] = cis_results_control_4alleles['gene_id'].str.split('DM.').str[1].str.split('G').str[0]
if cis_results_control_4alleles['chromosome'].isna().any():
    cis_results_control_4alleles.loc[cis_results_control_4alleles['chromosome'].isna(), 'chromosome'] = cis_results_control_4alleles.loc[cis_results_control_4alleles['chromosome'].isna(), 'gene_id'].str.extract(r'Sotub(\d+)g\d+')[0]

cis_results_control_4alleles['chr_hap'] = cis_results_control_4alleles['chromosome'] + "." + cis_results_control_4alleles['allele'].str[-1]

# Drop rows where chr_hap doesn't match the expected digit.digit pattern
cis_results_control_4alleles = cis_results_control_4alleles[~cis_results_control_4alleles['chr_hap'].str.contains(r'[^0-9\.1-4]')]
cis_results_control_4alleles['significant'] = (cis_results_control_4alleles['FDR'] < 0.05) & (cis_results_control_4alleles['ratio_difference'] > 0.1)

# Add a temporary column to track original order
cis_results_control_4alleles['plot_order'] = range(len(cis_results_control_4alleles))

# Create the swarm plot
fig, ax = plt.subplots(figsize=(30, 4))
sns.swarmplot(y=cis_results_control_4alleles['ratios_control_mean'],
              x=cis_results_control_4alleles['chr_hap'],
              hue=cis_results_control_4alleles['significant'],
              size=2,
              ax=ax,
            palette={True: 'red', False: 'grey'},
              order = ['01.1', '01.2', '01.3', '01.4',
         '02.1', '02.2', '02.3', '02.4',
         '03.1', '03.2', '03.3', '03.4',
         '04.1', '04.2', '04.3', '04.4',
         '05.1', '05.2', '05.3', '05.4',
         '06.1', '06.2', '06.3', '06.4',
         '07.1', '07.2', '07.3', '07.4',
         '08.1', '08.2', '08.3', '08.4',
         '09.1', '09.2', '09.3', '09.4',
         '10.1', '10.2', '10.3', '10.4',
         '11.1', '11.2', '11.3', '11.4',
         '12.1', '12.2', '12.3', '12.4'])


# Extract point positions from the plot
points_data = []
for collection in ax.collections:
    offsets = collection.get_offsets().data
    for x, y in offsets:
        points_data.append({'x': x, 'y': y})

# Match points to data by y-values (within tolerance)
point_positions = {}
for idx, row in cis_results_control_4alleles.iterrows():
    y_val = row['ratios_control_mean']
    allele_idx = ['hap1', 'hap2', 'hap3', 'hap4'].index(row['allele'])
    chr_hap  = ['01.1', '01.2', '01.3', '01.4',
         '02.1', '02.2', '02.3', '02.4',
         '03.1', '03.2', '03.3', '03.4',
         '04.1', '04.2', '04.3', '04.4',
         '05.1', '05.2', '05.3', '05.4',
         '06.1', '06.2', '06.3', '06.4',
         '07.1', '07.2', '07.3', '07.4',
         '08.1', '08.2', '08.3', '08.4',
         '09.1', '09.2', '09.3', '09.4',
         '10.1', '10.2', '10.3', '10.4',
         '11.1', '11.2', '11.3', '11.4',
         '12.1', '12.2', '12.3', '12.4'].index(row['chr_hap'])

    # Find matching point (closest in y-value near the correct x position)
    for point in points_data:
        if abs(point['y'] - y_val) < 0.001 and abs(point['x'] - chr_hap) < 0.5:
            if idx not in point_positions:
                point_positions[idx] = (point['x'], point['y'])
                break

# Connect paired points by Synt_id
for synt_id in cis_results_control_4alleles['Synt_id'].unique():
    subset = cis_results_control_4alleles[cis_results_control_4alleles['Synt_id'] == synt_id]
    if len(subset) >= 2:  # Connect all points with same Synt_id
        indices = subset.index.tolist()
        valid_indices = [i for i in indices if i in point_positions]

        if len(valid_indices) >= 2:
            for i in range(len(valid_indices) - 1):
                x_vals = [point_positions[valid_indices[i]][0], point_positions[valid_indices[i+1]][0]]
                y_vals = [point_positions[valid_indices[i]][1], point_positions[valid_indices[i+1]][1]]
                ax.plot(x_vals, y_vals, color='gray', linewidth=0.5, alpha=0.2, zorder=0)

plt.yticks([0.0, 0.15, 0.25, 0.35, 0.5, 1.0])
plt.xticks(ticks=range(48), labels=np.tile(np.arange(1, 5), 12))
plt.axhline(y=0.35, color='r', linestyle='--')
plt.axhline(y=0.15, color='r', linestyle='--')

# Add chromosome labels with group separators below x-axis
from matplotlib import transforms
trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)
n_chrs, n_alleles = 12, 4
chr_centers   = np.array([i * n_alleles + 1.5 for i in range(n_chrs)])
sep_positions = np.array([n_alleles * (i + 1) - 0.5 for i in range(n_chrs - 1)])

for i, pos in enumerate(chr_centers):
    ax.text(pos, -0.25, f'chr{i+1:02d}', transform=trans,
            ha='center', va='bottom', fontsize=14, clip_on=False)

for pos in sep_positions:
    ax.plot([pos, pos], [0, -0.20], transform=trans,
            color='black', lw=0.8, clip_on=False)

plt.tight_layout()
#plt.savefig("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/allelic_ratios_swarm_connected.svg", bbox_inches='tight')
plt.show()

In [ ]:
# cis_results_control_unique.to_csv("LR_DESIREE_PAPER/ANALYSIS/ASE_results/short_cis_results_1mm_unique_counts_control.tsv", index=False)

cis_results_control_em.to_csv("LR_DESIREE_PAPER/ANALYSIS/final_tables_paper/short_cis_results_1mm_em_counts_control.tsv", index=False)

In [ ]:

poly.plot_top_differential_syntelogs(
cis_results_control_em, 
                                           n = 20, 
                                           sort_by='ratio_difference')

#### Plot ratios and CPM for one particular gene

In [ ]:
gene_id = "Synt_id_62117"
#Soltu.DM.12G007300
# filter results for specific synt id
cis_results_filtered_synt = cis_results_control_em[cis_results_control_em['Synt_id'].str.contains(gene_id)]
ax1 = poly.plot_top_differential_syntelogs(cis_results_filtered_synt, n = 20, figsize = (12, 3),sort_by='ratio_difference')

# filter results for specific synt id
cis_results_filtered_synt = cis_results_control_em[cis_results_control_em['Synt_id'].str.contains(gene_id)]
ax2 = poly.plot_top_differential_syntelogs(cis_results_filtered_synt, n = 20, figsize = (12, 3*1),sort_by='ratio_difference', plot_type='cpm' )